In [ ]:
%pip install gymnasium numpy pandas
%pip install torch stable-baselines3
%pip install openai


In [ ]:
import gymnasium as gym
from gymnasium import spaces
from dataclasses import dataclass

In [ ]:
@dataclass
class SpellingBeeConfig:
    center_letter: str
    letter_list: set[str]
    word_list: set[str]
    max_guesses: int = 10  


    def validate(self):
        assert self.center_letter in self.letter_list, "Center letter must be in the letter list"
        assert all(letter in self.letter_list for word in self.word_list for letter in word), "All words must be formed from the letter list"
        for word in self.word_list:
            assert any(letter == self.center_letter for letter in word), "All words must contain the center letter"
        assert max_guesses > 0, "Max guesses must be a positive integer"

In [ ]:
class SpellingBeeEnv(gym.Env):

    def __init__(self, config: SpellingBeeConfig):
        super().__init__()
        self.config = config
        self.num_guesses = 0
        self.total_points = 0
        self.words_guessed = set()
        self.info= {'history': []}  # To track the history of guesses and rewards

    def reset(self):
        self.num_guesses = 0
        self.total_points = 0
        self.words_guessed = set()
        self.info = {'history': []}  # To track the history of guesses and rewards
        return self._get_obs(), self.info  # Return initial observation and info

    def _get_obs(self):
        obs = {
            'num_guesses': self.num_guesses,
            'total_points': self.total_points,
            'words_guessed': list(self.words_guessed)
        }
        return obs

    def step(self, action):
        self.num_guesses += 1
        self.words_guessed.add(action)
        if not self._word_is_valid(action):
            self.info['history'].append((action, 0))
            return self._get_obs(), 0, self._is_done(), self.info
        
        reward = self._get_reward(action)
        self.total_points += reward
        self.info['history'].append((action, reward))
        return self._get_obs(), reward, self._is_done(), self.info  # Return a dummy observation, reward, done, and info
  
    def render(self, mode='human'):
        pass  # No rendering needed for this environment

    def close(self):
        pass  # No cleanup needed for this environment

    def _get_reward(self, word):
        reward = 0
        if len(word) <4:
            return 0
        elif len(word) == 4:
            return 1
        else:
            reward += len(word) 
            if self._is_pangram(word):
                reward += 7
            return reward
            
    def _is_pangram(self, word):
        return set(word) >= set(self.config.letter_list) 

    def _word_is_valid(self, word):
        return (word in self.config.word_list and 
                set(word) <= set(self.config.letter_list) and 
                self.config.center_letter in word)

    def _is_done(self):
        return self.num_guesses >= self.config.max_guesses



In [ ]:
center_letter= "N"
letter_list: set[str] = {"T","A","F","M","O","R","N"}
word_list: set[str] = {"FRONTMAN","FONT","AFFRONT", "MAROON","TORN"}
max_guesses: int = 10

In [ ]:
config = SpellingBeeConfig(center_letter, letter_list, word_list, max_guesses)
config.validate()  # Validate the configuration

In [ ]:
env = SpellingBeeEnv(config)
while not env._is_done():
    guess = input("Enter your guess: ").strip().upper()
    obs, reward, done, info = env.step(guess)
    print(f"Reward: {reward}, Total Points: {obs['total_points']}, Num Guesses: {obs['num_guesses']}")
    if done:
        print("Game Over!")
        break